# WeatherLSTM Training

Kompakter End-to-End-Ablauf: Daten laden, Sequenzen bauen, WeatherLSTM trainieren und gegen Persistence vergleichen.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from train_weather_rnn import Config, train
from weather_dataloader import make_weather_dataloaders

PROJECT_ROOT

## Daten pruefen

Feature-Legende:

| Feature | Bedeutung |
|---|---|
| `TT_TU` | Tagesmittel der Lufttemperatur [C] |
| `RF_TU` | Tagesmittel der relativen Luftfeuchte [%] |
| `R1` | Tagessumme des Niederschlags [mm] |
| `SD_SO` | Tagessumme der Sonnenscheindauer |
| `F` | Tagesmittel der Windgeschwindigkeit [m/s] |
| `P0` | Tagesmittel des Luftdrucks auf Stationshoehe [hPa] |
| `day_sin` | Sinus-Codierung des Tags im Jahr, bildet Saisonalitaet ab |
| `day_cos` | Cosinus-Codierung des Tags im Jahr, bildet Saisonalitaet ab |

In [ ]:
cfg = Config(
    data_path=str(PROJECT_ROOT / "data" / "processed" / "weather_clean_daily.csv"),
    output_dir=str(PROJECT_ROOT / "outputs" / "weather_lstm"),
    device="auto",  # Uses Apple MPS if available, otherwise CUDA, otherwise CPU.
)

data = make_weather_dataloaders(
    data_path=cfg.data_path,
    target=cfg.target,
    sequence_length=cfg.sequence_length,
    horizon=cfg.horizon,
    batch_size=cfg.batch_size,
    train_until=cfg.train_until,
    val_until=cfg.val_until,
    add_engineered_features=cfg.add_engineered_features,
)

x_batch, y_batch, dates = next(iter(data.train_loader))
print("Features:", data.feature_columns)
print("Train/Val/Test:", data.train_samples, data.val_samples, data.test_samples)
print("Batch X:", tuple(x_batch.shape), "Batch y:", tuple(y_batch.shape))
data.dataframe.head()

## Trainieren

In [ ]:
metrics = train(cfg)
pd.Series(metrics)

## Evaluation

In [ ]:
output_dir = Path(cfg.output_dir)
history = pd.read_csv(output_dir / "history.csv")
predictions = pd.read_csv(output_dir / "test_predictions.csv", parse_dates=["datetime"])

comparison = pd.DataFrame(
    [
        {"model": "WeatherLSTM", "RMSE": metrics["test_rmse"], "MAE": metrics["test_mae"]},
        {"model": "Persistence", "RMSE": metrics["persistence_test_rmse"], "MAE": metrics["persistence_test_mae"]},
    ]
)
display(comparison.round(3))

improvement = (metrics["persistence_test_rmse"] - metrics["test_rmse"]) / metrics["persistence_test_rmse"] * 100
print(f"RMSE improvement vs. persistence: {improvement:.1f}%")

## Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

axes[0].plot(history["epoch"], history["train_loss"], label="Train")
axes[0].plot(history["epoch"], history["val_loss"], label="Validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

focus = predictions.head(120)
axes[1].plot(focus["datetime"], focus["actual"], label="Actual", linewidth=2)
axes[1].plot(focus["datetime"], focus["prediction"], label="WeatherLSTM")
axes[1].plot(focus["datetime"], focus["persistence"], label="Persistence", alpha=0.75)
axes[1].set_title("First 120 test days")
axes[1].set_ylabel("Temperature [C]")
axes[1].legend()

plt.tight_layout()